# Resonator IQ fitting
This notebook shows how to fit complex S21 sweeps of a KID resonator with `citkid.res`, using synthetic data. It goes with `documents/res_fitting.pdf` (Appendix B of the Foote thesis), and uses the same notation.

The fitting model (Eq. B.10) is

$$S_{21}(f) = (I_0 + jQ_0)\, e^{-2\pi j (f - f_r)\tau} \left[1 - \frac{Q_r}{Q_c \cos\phi}\,\frac{e^{j\phi}}{1 + 2jy}\right],$$

where $y$ is the solution of $y - \frac{a}{1 + 4y^2} = y_0$, with $y_0 = Q_r (f - f_r)/f_r$ (Eq. B.6). The eight fit parameters, in citkid's order, are

| parameter | name in citkid | meaning |
|---|---|---|
| $f_r$ | `fr` | resonance frequency in the low-power limit (Hz) |
| $Q_r$ | `Qr` | total quality factor, $1/Q_r = 1/Q_i + 1/Q_c$ |
| $Q_r/Q_c$ | `amp` | diameter of the resonance circle, relative to the off-resonance transmission |
| $\phi$ | `phi` | impedance-mismatch rotation (rad) |
| $a_\mathrm{nl}$ | `a` | nonlinearity parameter. Bifurcation at $4\sqrt{3}/9 \approx 0.77$ |
| $I_0, Q_0$ | `i0`, `q0` | complex gain |
| $\tau$ | `tau` | cable delay (s) |

Sections:
1. The model, `citkid.res.funcs.nonlinear_iq`
2. A synthetic target sweep
3. Initial guesses, `citkid.res.guess`
4. Fitting with `citkid.res.fitter.fit_nonlinear_iq`
5. $Q_i$ and $Q_c$ with `citkid.res.util.qi_qc`
6. Fit options
7. Masking a neighboring resonance
8. A bifurcated resonator
9. Fitting several resonators at once, `citkid.res.multifit`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from citkid.res.funcs import nonlinear_iq, product_s21
from citkid.res.guess import guess_p0_pole, guess_p0_nonlinear_iq
from citkid.res.fitter import fit_nonlinear_iq
from citkid.res.util import qi_qc
from citkid.res.mask import nearby_tone_mask
from citkid.res.multifit import fit_poles, fit_resonators, multifit_model

names = ['fr', 'Qr', 'amp', 'phi', 'a', 'i0', 'q0', 'tau']

def print_fit(p_true, popt, perr = None, p0 = None):
    """Print injected and fitted parameters, and the pull (fit - true) / err."""
    header = f"{'':>4} {'true':>13} {'fit':>13}"
    header += f" {'err':>10} {'pull':>6}" if perr is not None else ''
    header += f" {'guess':>13}" if p0 is not None else ''
    print(header)
    for i, name in enumerate(names[:len(p_true)]):
        line = f'{name:>4} {p_true[i]:13.9g} {popt[i]:13.9g}'
        if perr is not None:
            pull = (popt[i] - p_true[i]) / perr[i] if perr[i] > 0 else np.nan
            line += f' {perr[i]:10.3g} {pull:6.2f}'
        if p0 is not None:
            line += f' {p0[i]:13.9g}'
        print(line)

## 1. The model
`nonlinear_iq(f, fr, Qr, amp, phi, a, i0, q0, tau, downward)` evaluates Eq. B.10. It is compiled with numba, so every argument must be given positionally with the right type: `f` a float64 array, the parameters floats, and `downward` a bool.

`downward` sets the sweep direction. Above bifurcation the cubic for $y$ has three real roots in a range of $y_0$, and which stable root the resonator sits on depends on whether the tone is swept down (decreasing frequency, `downward = True`, the default throughout citkid) or up. Below bifurcation the two directions give the same result.

In [ ]:
fr, Qr, amp, phi = 1e9, 2e4, 0.6, 0.
span = 6 * fr / Qr
f = np.linspace(fr - span / 2, fr + span / 2, 2000)

fig, axs = plt.subplots(1, 2, figsize = [10, 4], layout = 'tight')
axs[0].set(xlabel = 'I', ylabel = 'Q', aspect = 'equal')
axs[1].set(xlabel = r'$f - f_r$ (kHz)', ylabel = r'$|S_{21}|$ (dB)')
for a in [0, 0.25, 0.5, 0.75, 1.0]:
    z = nonlinear_iq(f, fr, Qr, amp, phi, a, 1., 0., 0., True)
    axs[0].plot(z.real, z.imag, label = f'a = {a}')
    axs[1].plot((f - fr) / 1e3, 20 * np.log10(np.abs(z)))
axs[0].legend(loc = 'center')
axs[1].set_title('downward sweep')
plt.show()

Above bifurcation the two sweep directions differ: the downward sweep follows the resonance down in frequency and jumps where its branch ends.

In [ ]:
a = 1.5
fig, ax = plt.subplots(figsize = [5, 4], layout = 'tight')
ax.set(xlabel = r'$f - f_r$ (kHz)', ylabel = r'$|S_{21}|$ (dB)',
       title = f'a = {a}')
for downward in [True, False]:
    z = nonlinear_iq(f, fr, Qr, amp, phi, a, 1., 0., 0., downward)
    label = 'downward' if downward else 'upward'
    ax.plot((f - fr) / 1e3, 20 * np.log10(np.abs(z)), label = label)
ax.legend()
plt.show()

## 2. A synthetic target sweep
We simulate a target sweep around one resonance with $Q_i = 2\times10^5$, $Q_c = 5\times10^4$, a mild nonlinearity, a complex gain and a cable delay, and add complex Gaussian noise with a fixed seed. `fit_nonlinear_iq` works best on a span of about $6 f_r / Q_r$ with ~500 points (200 still works).

In [ ]:
rng = np.random.default_rng(0)

fr, Qi, Qc, phi, a = 500e6, 2e5, 5e4, 0.2, 0.3
Qr = 1 / (1 / Qi + 1 / Qc)
amp = Qr / Qc
gain = 0.8 * np.exp(0.6j) # i0 + j q0
tau = 20e-9
p_true = [fr, Qr, amp, phi, a, gain.real, gain.imag, tau]

span = 6 * fr / Qr
f = np.linspace(fr - span / 2, fr + span / 2, 500)
noise = 2e-3 # per quadrature
z = nonlinear_iq(f, *p_true, True)
z = z + noise * (rng.normal(size = f.size) + 1j * rng.normal(size = f.size))

fig, axs = plt.subplots(1, 2, figsize = [10, 4], layout = 'tight')
axs[0].plot(z.real, z.imag, '.', ms = 3)
axs[0].set(xlabel = 'I', ylabel = 'Q', aspect = 'equal')
axs[1].plot((f - fr) / 1e3, 20 * np.log10(np.abs(z)), '.', ms = 3)
axs[1].set(xlabel = r'$f - f_r$ (kHz)', ylabel = r'$|S_{21}|$ (dB)')
plt.show()

## 3. Initial guesses
`citkid.res.guess` has two guess functions. Both return `p0 = [fr, Qr, amp, phi, a, i0, q0, tau]`.

- `guess_p0_pole(f, z)` (the default in `fit_nonlinear_iq`) fits one pole plus a linear complex gain polynomial (`multifit.fit_poles`). It does not need the gain removed, and it adds a second pole if a neighboring resonance appears near the edge of the sweep. Its `a` guess is always 0.
- `guess_p0_nonlinear_iq(f, z)` is the procedure in Section B.1.1 of the document (circle fit for `phi` and `amp`, FWHM for `Qr`, the steepest IQ step for `a`). It assumes the gain and cable delay were removed first, so its `tau` guess is 0.

In [ ]:
p0_pole = guess_p0_pole(f, z)
p0_nl = guess_p0_nonlinear_iq(f, z)
print(f"{'':>4} {'true':>13} {'guess_p0_pole':>14} {'guess_p0_nl_iq':>14}")
for name, t, g1, g2 in zip(names, p_true, p0_pole, p0_nl):
    print(f'{name:>4} {t:13.9g} {g1:14.9g} {g2:14.9g}')

## 4. Fitting
`fit_nonlinear_iq(f, z)` sorts the data, makes the guess (if `p0` is None), sets default bounds, and fits with `scipy.optimize.curve_fit` (trust region reflective), refitting from the result if the normalized residual is large. It returns

- `p0`: the initial guess,
- `popt`, `perr`: the fit parameters and their 1-sigma errors,
- `nrmse` = $\|z - z_\mathrm{fit}\|^2 / \|z\|^2$, a goodness-of-fit measure (comparable between sweeps when $|I_0 + jQ_0| \approx 1$; see Section B.1.3),
- `(fig, ax)`: the plot of the data and fit if `plotq = True`, else `(None, None)`.

The pull column is (fit - true) / err, which should be of order 1.

In [ ]:
p0, popt, perr, nrmse, (fig, axs) = fit_nonlinear_iq(f, z, plotq = True)
print(f'nrmse = {nrmse:.2e}')
print_fit(p_true, popt, perr)

## 5. Internal and coupling quality factors
The fit gives $Q_r$ and `amp` $= Q_r / Q_c$. `qi_qc(Qr, amp, Qr_err, amp_err)` converts them to $Q_i = Q_r / (1 - \mathrm{amp})$ and $Q_c = Q_r / \mathrm{amp}$, with errors propagated to first order (ignoring the $Q_r$-amp covariance). It works on arrays, giving NaN for invalid inputs and $Q_i = \infty$ for amp at its upper bound $1 - 10^{-6}$.

In [ ]:
Qi_fit, Qc_fit, Qi_err, Qc_err = qi_qc(popt[1], popt[2], perr[1], perr[2])
print(f'Qi = {Qi_fit:.0f} +- {Qi_err:.0f} (true {Qi:.0f})')
print(f'Qc = {Qc_fit:.0f} +- {Qc_err:.0f} (true {Qc:.0f})')

## 6. Fit options
- `p0`: a full initial guess, which replaces the automatic one. `fr_guess` and `tau_guess` override only `p0[0]` and `p0[7]`.
- `fit_tau = False`: hold `tau` fixed at `p0[7]` (or `tau_guess`), which is faster. Its error is returned as 0. This is useful when the cable delay is known, e.g. from a gain fit to a wider sweep.
- `bounds`: `(lower, upper)` lists of 8 values. The defaults keep `fr` within the data, `Qr` within a factor 10 of the guess, `a` in [0, 1] and `tau` within $\pm 1\,\mu$s.
- `downward`: the sweep direction of the model (default True).

Here we fix the cable delay at its known value:

In [ ]:
p0, popt, perr, nrmse, _ = fit_nonlinear_iq(f, z, fit_tau = False,
                                            tau_guess = tau)
print(f'nrmse = {nrmse:.2e}')
print_fit(p_true, popt, perr)

## 7. Masking a neighboring resonance
`fit_nonlinear_iq` fits one resonance, so a neighbor inside the sweep biases it. To simulate one, we multiply the sweep by the transmission of a second resonator 2.5 linewidths above, using `product_s21` (the product of single-resonator transmissions, without gain).

There is no mask argument to the fitter: select the points to fit before calling it. `citkid.res.mask.nearby_tone_mask(f, f_tone, f_others)` cuts the sweep at the midpoint between this tone and the nearest other tone on each side (here the other tone sits on the neighboring resonance).

In [ ]:
lw = fr / Qr # linewidth
fr_n = fr + 2.5 * lw
z_n = z * product_s21(f, [fr_n], [3e4], [0.6], [0.])

mask = nearby_tone_mask(f, fr, [fr_n])

fig, axs = plt.subplots(1, 2, figsize = [10, 4], layout = 'tight')
axs[0].plot(z_n.real, z_n.imag, '.', ms = 3, color = '0.7', label = 'masked')
axs[0].plot(z_n[mask].real, z_n[mask].imag, '.', ms = 3, label = 'fit')
axs[0].set(xlabel = 'I', ylabel = 'Q', aspect = 'equal')
axs[0].legend()
axs[1].plot((f - fr) / 1e3, 20 * np.log10(np.abs(z_n)), '.', ms = 3,
            color = '0.7')
axs[1].plot((f[mask] - fr) / 1e3, 20 * np.log10(np.abs(z_n[mask])), '.', ms = 3)
axs[1].set(xlabel = r'$f - f_r$ (kHz)', ylabel = r'$|S_{21}|$ (dB)')
plt.show()

In [ ]:
_, popt_full, perr_full, nrmse_full, _ = fit_nonlinear_iq(f, z_n)
_, popt_mask, perr_mask, nrmse_mask, _ = fit_nonlinear_iq(f[mask], z_n[mask])
print(f'nrmse: full sweep {nrmse_full:.1e}, masked {nrmse_mask:.1e}')
print(f"{'':>4} {'true':>12} {'full sweep':>12} {'masked':>12}")
for i, name in enumerate(names[:5]):
    print(f'{name:>4} {p_true[i]:12.9g} {popt_full[i]:12.9g} '
          f'{popt_mask[i]:12.9g}')

Masking removes most of the bias. The neighbor's tail still reaches into the masked sweep, so the masked fit is close but not exact; for strongly overlapping resonances, fit them together (Section 9).

## 8. A bifurcated resonator
With $a = 0.9 > 0.77$ the downward sweep has a jump. The fit still recovers all parameters, as long as the model uses the same sweep direction as the data. Fitting with the wrong direction (`downward = False`) gives a much worse residual.

In [ ]:
p_bif = list(p_true)
p_bif[4] = 0.9
z_bif = nonlinear_iq(f, *p_bif, True)
z_bif = z_bif + noise * (rng.normal(size = f.size)
                         + 1j * rng.normal(size = f.size))

p0, popt, perr, nrmse, (fig, axs) = fit_nonlinear_iq(f, z_bif, plotq = True)
print(f'nrmse = {nrmse:.2e}')
print_fit(p_bif, popt, perr)

_, _, _, nrmse_up, _ = fit_nonlinear_iq(f, z_bif, downward = False)
print(f'\nnrmse with the wrong sweep direction: {nrmse_up:.2e}')

A fitted `a` above 0.77 means the resonator is bifurcated. The default bounds cap `a` at 1: strongly bifurcated resonators (true `a` well above 1) cannot be fit well and end near the bound with a large `nrmse`. Lower the readout power instead.

## 9. Fitting several resonators at once
`citkid.res.multifit` fits several resonators in one window of a sweep (for example close pairs, or a window of a full VNA sweep). The model is a complex gain polynomial times `product_s21`, the product of single-resonator transmissions. It runs in two stages:

- `fit_poles(f, z, fr0)` (Stage A) needs only frequency guesses `fr0`. It fits the window as a gain polynomial plus one pole per resonator, solving the linear parts directly, and returns a dict with `fr`, `Qr`, `amp`, `phi` (arrays), `fr_err`, `Qr_err`, the gain polynomial, and `nrmse`. It doesn't fit the nonlinearity.
- `fit_resonators(f, z, fr0, Qr0, amp0, phi0)` (Stage B) refines a Stage A result with the product model, returns errors for all parameters, and with `fit_nonlinear = True` also fits `a` for each resonator.

`multifit_model(f, result)` evaluates either result. Here two resonators are 2 linewidths apart, the lower one nonlinear, under a gain with a slope and curvature.

In [ ]:
rng = np.random.default_rng(1)
fr2 = np.array([500e6, 500e6 + 2 * lw])
Qr2 = np.array([4e4, 3e4])
amp2 = np.array([0.8, 0.5])
phi2 = np.array([0.2, -0.3])
a2 = np.array([0.3, 0.])

f_w = np.linspace(fr2[0] - 5 * lw, fr2[1] + 5 * lw, 1000)
x = (f_w - f_w.mean()) / np.ptp(f_w)
gain_w = gain * (1 + 0.05 * x + 0.02j * x ** 2) \
    * np.exp(-2j * np.pi * (f_w - fr2[0]) * tau)
z_w = gain_w * product_s21(f_w, fr2, Qr2, amp2, phi2, a2)
z_w = z_w + noise * (rng.normal(size = f_w.size)
                     + 1j * rng.normal(size = f_w.size))

# Stage A: start from rough frequency guesses only
fr0 = fr2 + np.array([-0.3, 0.3]) * lw
res_a = fit_poles(f_w, z_w, fr0)
# Stage B: refine, including the nonlinearity
res_b = fit_resonators(f_w, z_w, res_a['fr'], res_a['Qr'], res_a['amp'],
                       res_a['phi'], fit_nonlinear = True)
print(f"Stage A nrmse {res_a['nrmse']:.2e}, "
      f"Stage B nrmse {res_b['nrmse']:.2e}\n")
print(f"{'':>7} {'true':>12} {'Stage A':>12} {'Stage B':>12} {'err B':>9}")
for key, true in zip(['fr', 'Qr', 'amp', 'phi', 'a'],
                     [fr2, Qr2, amp2, phi2, a2]):
    for i in range(2):
        val_a = res_a[key][i] if key in res_a else np.nan
        print(f"{key + f'[{i}]':>7} {true[i]:12.9g} {val_a:12.9g} "
              f"{res_b[key][i]:12.9g} {res_b[key + '_err'][i]:9.2g}")

In [ ]:
z_model = multifit_model(f_w, res_b)
fig, axs = plt.subplots(1, 2, figsize = [10, 4], layout = 'tight')
axs[0].plot(z_w.real, z_w.imag, '.', ms = 3, label = 'data')
axs[0].plot(z_model.real, z_model.imag, '--', label = 'Stage B fit')
axs[0].set(xlabel = 'I', ylabel = 'Q', aspect = 'equal')
axs[0].legend()
axs[1].plot((f_w - fr2[0]) / 1e3, 20 * np.log10(np.abs(z_w)), '.', ms = 3)
axs[1].plot((f_w - fr2[0]) / 1e3, 20 * np.log10(np.abs(z_model)), '--')
axs[1].set(xlabel = r'$f - f_{r,0}$ (kHz)', ylabel = r'$|S_{21}|$ (dB)')
plt.show()